# Challenge 4: Quantum Correlation Switch
Qiskit Fall Fest 2026 – IBM Quantum, Vishnu Institute of Technology

In [ ]:
import numpy as np
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram

try:
    from qiskit_aer import AerSimulator
    backend = AerSimulator()
except ImportError:
    from qiskit.providers.basic_provider import BasicSimulator
    backend = BasicSimulator()

SHOTS = 100

# Gate colours: [fill, text]
circuit_style = {
    "displaycolor": {
        "h":       ["#003399", "#FFFFFF"],
        "cx":      ["#003399", "#FFFFFF"],
        "x":       ["#CC3300", "#FFFFFF"],   # the "switch" gate
        "measure": ["#A9A9A9", "#000000"],
    }
}

## Predictions (before running)
- **Part A (correlated):** `00` and `11` should dominate, about 50% each. `01` and `10` should not appear, because after entanglement the two qubits always agree.
- **Part B (opposite):** `01` and `10` should dominate, about 50% each. `00` and `11` should not appear, because an X gate on qubit 1 flips its result so the qubits always disagree.
- **Will all four outcomes appear?** No. Only two of the four are allowed in each circuit, because the qubits are entangled. Each of the two allowed outcomes is about 50% (not exactly, because of shot noise with 100 shots).

## Part A – Correlated outcomes (H + CX)

In [ ]:
qc_A = QuantumCircuit(2, 2)
qc_A.h(0)                 # put qubit 0 in superposition
qc_A.cx(0, 1)             # entangle: qubit 1 copies qubit 0
qc_A.measure([0, 1], [0, 1])

qc_A.draw(output="mpl", style=circuit_style)

## Part B – Opposite outcomes (H + CX + X)

In [ ]:
qc_B = QuantumCircuit(2, 2)
qc_B.h(0)
qc_B.cx(0, 1)
qc_B.x(1)                 # the "switch": flips qubit 1
qc_B.measure([0, 1], [0, 1])

qc_B.draw(output="mpl", style=circuit_style)

## Run both circuits (100 shots each)

In [ ]:
counts_A = backend.run(transpile(qc_A, backend), shots=SHOTS).result().get_counts()
counts_B = backend.run(transpile(qc_B, backend), shots=SHOTS).result().get_counts()

print("Part A results:", counts_A)
print("Part B results:", counts_B)

## Histograms

In [ ]:
plot_histogram(counts_A, title="Part A: Correlated (00 and 11)",
               color="#003399", figsize=(7, 5))

In [ ]:
plot_histogram(counts_B, title="Part B: Opposite (01 and 10)",
               color="#CC3300", figsize=(7, 5))

## Part C – Comparison

In [ ]:
plot_histogram([counts_A, counts_B],
               legend=["Part A (H, CX)", "Part B (H, CX, X)"],
               color=["#003399", "#CC3300"],
               title="Correlation Switch: A vs B", figsize=(8, 5))

In [ ]:
def pct(counts, keys):
    return sum(counts.get(k, 0) for k in keys) / SHOTS * 100

print(f"Part A: same-bit outcomes (00, 11)     = {pct(counts_A, ['00','11']):.0f}%")
print(f"Part A: different-bit outcomes (01,10) = {pct(counts_A, ['01','10']):.0f}%")
print(f"Part B: same-bit outcomes (00, 11)     = {pct(counts_B, ['00','11']):.0f}%")
print(f"Part B: different-bit outcomes (01,10) = {pct(counts_B, ['01','10']):.0f}%")

## Explanation
- **Dominant in Part A:** `00` and `11` (about 50% each).
- **Dominant in Part B:** `01` and `10` (about 50% each).
- **What changed:** one extra X gate on qubit 1, added after the CNOT. Part A uses 2 gates (H, CX) and Part B uses 3 (H, CX, X), within the limit of 4.
- **Why the relationship changed:** H puts qubit 0 in superposition and CX entangles the qubits into the Bell state (|00⟩ + |11⟩)/√2, so they always agree. The X gate swaps |0⟩ and |1⟩ on qubit 1, turning the state into (|01⟩ + |10⟩)/√2, so the qubits always disagree. The qubits are still perfectly correlated; only the *type* of correlation switched (same → opposite). Each individual result is random (50/50), but the relationship between the two qubits is fixed, and the X gate acts as the switch.